# SILVER LAYER SCRIPT

## Data Access using App

### Use your credentials here

## Importing 

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

## Data loading 

### Reading Data

In [0]:
df_cal = spark.read.format("csv")\
    .option("header",True)\
    .option("inferSchema",True)\
    .load("abfss://bronze@awdatalakepro1.dfs.core.windows.net/AdventureWorks_Calendar")


In [0]:
df_cal.display()

In [0]:
df_cus = spark.read.format('csv')\
            .option("header",True)\
            .option("inferSchema",True)\
            .load('abfss://bronze@awdatalakepro1.dfs.core.windows.net/AdventureWorks_Customers')

In [0]:
df_cus.display()

In [0]:
df_procat = spark.read.format('csv')\
            .option("header",True)\
            .option("inferSchema",True)\
            .load('abfss://bronze@awdatalakepro1.dfs.core.windows.net/AdventureWorks_Product_Categories')

In [0]:
df_procat.display()

In [0]:
df_pro = spark.read.format('csv')\
            .option("header",True)\
            .option("inferSchema",True)\
            .load('abfss://bronze@awdatalakepro1.dfs.core.windows.net/AdventureWorks_Products')

In [0]:
df_pro.display()

In [0]:
df_ret = spark.read.format('csv')\
            .option("header",True)\
            .option("inferSchema",True)\
            .load('abfss://bronze@awdatalakepro1.dfs.core.windows.net/AdventureWorks_Returns')

In [0]:
df_ret.display()

In [0]:
df_sales = spark.read.format('csv')\
            .option("header",True)\
            .option("inferSchema",True)\
            .load('abfss://bronze@awdatalakepro1.dfs.core.windows.net/AdventureWorks_Sales*')

In [0]:
df_sales.display()

In [0]:
df_ter = spark.read.format('csv')\
            .option("header",True)\
            .option("inferSchema",True)\
            .load('abfss://bronze@awdatalakepro1.dfs.core.windows.net/AdventureWorks_Territories')

In [0]:
df_ter.display()

In [0]:
df_subcat = spark.read.format('csv')\
            .option("header",True)\
            .option("inferSchema",True)\
            .load('abfss://bronze@awdatalakepro1.dfs.core.windows.net/Product_Subcategories')

In [0]:
df_subcat.display()

##Transformations

### Calendar

In [0]:
df_cal.display()

In [0]:
df_cal = df_cal.withColumn('Month',month(col('Date')))\
        .withColumn('Year',year(col('Date')))

In [0]:
df_cal.display()

In [0]:
df_cal.write.format("Parquet")\
    .mode("append")\
    .option("path","abfss://silver@awdatalakepro1.dfs.core.windows.net/AdventureWorks_Calendar")\
    .save()

### Customer

In [0]:
df_cus.display()

In [0]:
df_cus.withColumn('FullName',concat(col('prefix'),lit(' '),col('Firstname'),lit(' '),col("LastName"))).display()

In [0]:
df_cus = df_cus.withColumn("Fullname",concat_ws(" ",col("prefix"),col("firstname"),col("lastname")))

In [0]:
df_cus.write.format("Parquet")\
    .mode("append")\
    .option("path","abfss://silver@awdatalakepro1.dfs.core.windows.net/AdventureWorks_Customers")\
    .save()


### Sub_categories

In [0]:
df_subcat.display()

In [0]:
df_subcat.write.format("parquet")\
    .mode("append")\
    .option("path","abfss://silver@awdatalakepro1.dfs.core.windows.net/AdventureWorks_SubCategories")\
    .save()

### Products

In [0]:
df_pro.display()

In [0]:
df_pro.withColumn('ProductSKU',split(col('ProductSKU'),'-')[0])\
        .withColumn('ProductName',split(col('ProductName'),' ')[0])

In [0]:
df_pro.write.format("parquet")\
    .mode("append")\
    .option("path","abfss://silver@awdatalakepro1.dfs.core.windows.net/AdventureWorks_Products")\
    .save()

### Returns

In [0]:
df_ret.write.format("parquet")\
    .mode("append")\
    .option("path","abfss://silver@awdatalakepro1.dfs.core.windows.net/AdventureWorks_Returns")\
    .save()

### Territories

In [0]:
df_ter.write.format("parquet")\
    .mode("append")\
    .option("path","abfss://silver@awdatalakepro1.dfs.core.windows.net/AdventureWorks_Territories")\
    .save()

### Sales

In [0]:
df_sales.display()

In [0]:
df_sales.withColumn('StockDate',to_timestamp(col('stockdate')))\
        .withColumn('Ordernumber',regexp_replace(col('ordernumber'),'S','T'))\
        .withColumn('multiply',col("OrderLineItem")*col("OrderQuantity")).display()

In [0]:
df_sales.write.format("parquet")\
    .mode("append")\
    .option("path","abfss://silver@awdatalakepro1.dfs.core.windows.net/AdventureWorks_Sales")\
    .save()

### Sales Analysis

In [0]:
df_sales.groupby("OrderDate").agg(count("OrderNumber").alias("Total_order")).display()

Databricks visualization. Run in Databricks to view.

### Product categories

In [0]:
df_procat.display()

In [0]:
df_procat.write.format("parquet")\
    .mode("append")\
    .option("path","abfss://silver@awdatalakepro1.dfs.core.windows.net/AdventureWorks_Product_Categories")\
    .save()